In [1]:
import os
import pandas as pd
import numpy as np

# Dataset used
### 1. goEmotions sentiment (filtered for fear, nervousness, sadness, grief, neutral)
### 2. Translated versions in Bengali, Chinese and Tamil
### 3. Sentiment-only training/evaluation for base model and LoRA adapters

In [24]:
df_sentiment_path = r"C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\goemotions_1.csv"
df_sentiment = pd.read_csv(df_sentiment_path, encoding='utf-8')

emotion_cols = ["fear", "grief", "nervousness", "sadness", "neutral"]
keep_emotion_mask = (df_sentiment[emotion_cols].sum(axis=1) > 0)

unclear_col = df_sentiment["example_very_unclear"]
if unclear_col.dtype == bool:
    unclear_mask = unclear_col
else:
    unclear_mask = unclear_col.astype(str).str.strip().str.lower().isin(["true", "1", "yes"] )

df_sentiment = df_sentiment[keep_emotion_mask & ~unclear_mask].copy()

# Convert one-hot emotion columns into a single label column
df_sentiment["label"] = df_sentiment[emotion_cols].eq(1).idxmax(axis=1)

# Drop the unclear flag and original one-hot columns
df_sentiment = df_sentiment.drop(columns=["example_very_unclear"] + emotion_cols)


# Final training dataset
df_sentiment_clean = df_sentiment[["text", "label"]].copy()
print(df_sentiment_clean['label'].value_counts())
df_sentiment_clean


label
neutral        18423
sadness         2043
fear            1048
nervousness      438
grief            220
Name: count, dtype: int64


,text,label
0,That game hurt.,sadness
1,"You do right, if you don't care then fuck 'em!",neutral
3,"[NAME] was nowhere near them, he was by the Fa...",neutral
9,"I have, and now that you mention it, I think t...",neutral
11,BUT IT'S HER TURN! /s,neutral
...,...,...
69987,"That would involve reading, and redhats aren't...",neutral
69992,"See her recent article in The Atlantic, in add...",neutral
69993,"Before we continue, I would first like to ask ...",neutral
69997,No but it should be,neutral


In [26]:
# Build sentiment-only datasets with per-language balancing target:
# 3000 rows each language, exactly 2000 neutral and 1000 non-neutral (with replacement if needed).
import re

TARGET_ROWS_PER_LANGUAGE = 3000
TARGET_NEUTRAL_ROWS = 2000
TARGET_EMOTION_ROWS = TARGET_ROWS_PER_LANGUAGE - TARGET_NEUTRAL_ROWS
languages = ["english", "bengali", "chinese", "tamil"]
non_english_languages = ["bengali", "chinese", "tamil"]


def clean_text_value(text):
    if pd.isna(text):
        return ""
    cleaned = str(text)

    # Remove common mojibake artifacts seen in this dataset.
    for token in ["»â€", "â™€ï", "â€™", "â€œ", "â€\x9d", "â€”", "â€“", "Â", "»", "âœ", "ðŸ", "€™", "'€", "�"]:
        cleaned = cleaned.replace(token, " ")

    cleaned = re.sub(r"âœ\s*€\s*ðŸ\s*'?€", " ", cleaned)
    cleaned = re.sub(r"(?:â[€€™œ80-9f]+|ðŸ[80-bf]*|Ã[80-bf]+)", " ", cleaned)

    cleaned = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


def clean_text_series(series):
    return series.map(clean_text_value)


def sample_exact_rows(df, target_rows, seed=42):
    if len(df) == 0:
        raise ValueError("Dataset became empty after cleaning/filtering.")

    replace = len(df) < target_rows
    return df.sample(n=target_rows, replace=replace, random_state=seed).reset_index(drop=True)


def split_with_english_remainder(df, target_rows, seed=42):
    shuffled = df.sample(frac=1, random_state=seed).reset_index(drop=True)
    needed_for_non_english = len(non_english_languages) * target_rows

    splits = {}
    if len(shuffled) >= needed_for_non_english:
        for i, lang in enumerate(non_english_languages):
            start = i * target_rows
            end = start + target_rows
            splits[lang] = shuffled.iloc[start:end].reset_index(drop=True)

        splits["english"] = shuffled.iloc[needed_for_non_english:].reset_index(drop=True)
    else:
        print(
            f"Warning: only {len(shuffled)} rows available; using all for english and sampling {target_rows} for each non-English language with replacement."
        )
        splits["english"] = shuffled.copy().reset_index(drop=True)
        for i, lang in enumerate(non_english_languages):
            splits[lang] = sample_exact_rows(shuffled, target_rows, seed=seed + i + 1)

    return splits


def rebalance_neutral_emotion(df, total_rows=3000, neutral_rows=2000, seed=42):
    if neutral_rows > total_rows:
        raise ValueError("neutral_rows cannot exceed total_rows.")

    emotion_rows = total_rows - neutral_rows
    work = df.copy().reset_index(drop=True)
    work["label"] = work["label"].astype(str)

    neutral_df = work[work["label"] == "neutral"].copy().reset_index(drop=True)
    emotion_df = work[work["label"] != "neutral"].copy().reset_index(drop=True)

    if len(neutral_df) == 0:
        raise ValueError("No neutral rows available; cannot satisfy target mix.")
    if len(emotion_df) == 0:
        raise ValueError("No non-neutral rows available; cannot satisfy target mix.")

    # Force exact target counts, oversampling with replacement when a class is short.
    sampled_neutral = sample_exact_rows(neutral_df, neutral_rows, seed=seed + 1)
    sampled_emotion = sample_exact_rows(emotion_df, emotion_rows, seed=seed + 2)

    out = pd.concat([sampled_neutral, sampled_emotion], ignore_index=True)
    out = out.sample(frac=1, random_state=seed + 3).reset_index(drop=True)
    return out


output_dir = os.path.join(os.path.dirname(df_sentiment_path), "data")
os.makedirs(output_dir, exist_ok=True)

# Sentiment only: text + label
sentiment_task_df = df_sentiment_clean[["text", "label"]].copy()
sentiment_task_df["text"] = clean_text_series(sentiment_task_df["text"])
sentiment_task_df = sentiment_task_df[sentiment_task_df["text"].str.strip() != ""].copy()

sentiment_by_language = split_with_english_remainder(
    sentiment_task_df,
    target_rows=TARGET_ROWS_PER_LANGUAGE,
    seed=42,
)

# Enforce per-language target composition.
for i, lang in enumerate(languages):
    sentiment_by_language[lang] = rebalance_neutral_emotion(
        sentiment_by_language[lang],
        total_rows=TARGET_ROWS_PER_LANGUAGE,
        neutral_rows=TARGET_NEUTRAL_ROWS,
        seed=100 + i,
    )

for lang in languages:
    split_df = sentiment_by_language[lang]
    out_path = os.path.join(output_dir, f"sentiment_{lang}.xlsx")
    split_df.to_excel(out_path, index=False)
    print(f"Saved {len(split_df)} rows to {out_path}")

print("Sentiment split sizes:", {lang: len(df_part) for lang, df_part in sentiment_by_language.items()})
print("Target rows per language:", TARGET_ROWS_PER_LANGUAGE)
print("Target neutral rows per language:", TARGET_NEUTRAL_ROWS)
print("Target non-neutral rows per language:", TARGET_EMOTION_ROWS)

for lang in languages:
    counts = sentiment_by_language[lang]["label"].astype(str).eq("neutral").map({True: "neutral", False: "non_neutral"}).value_counts().to_dict()
    print(f"{lang} label counts:", counts)

sentiment_by_language["english"].head()

Saved 3000 rows to C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_english.xlsx
Saved 3000 rows to C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali.xlsx
Saved 3000 rows to C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_chinese.xlsx
Saved 3000 rows to C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_tamil.xlsx
Sentiment split sizes: {'bengali': 3000, 'chinese': 3000, 'tamil': 3000, 'english': 3000}
Target rows per language: 3000
Target neutral rows per language: 2000
Target non-neutral rows per language: 1000
english label counts: {'neutral': 2000, 'non_neutral': 1000}
bengali label counts: {'neutral': 2000, 'non_neutral': 1000}
chinese label counts: {'neutral': 2000, 'non_neutral': 1000}
tamil label counts: {'neutral': 2000, 'non_neutral': 1000}


,text,label
0,YPG has carried out numerous attacks. Yet you ...,sadness
1,Well that's why I'm reading up on the scientif...,neutral
2,"Yes, listening to loud noises can damage your ...",fear
3,"Whoa, should've said that in the post. Threats...",fear
4,I'm 25 and I still don't have a license. I'm t...,fear


# Translation to Bengali, Chinese and Tamil

In [27]:
import os
import time
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor

# Ensure deep-translator package is available in the notebook kernel.
try:
    from deep_translator import GoogleTranslator
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "deep-translator"])
    from deep_translator import GoogleTranslator

# Translate sentiment files in data/ from English into Bengali, Chinese and Tamil.
data_dir = os.path.join(os.path.dirname(df_sentiment_path), "data")

lang_codes = {
    "bengali": "bn",
    "chinese": "zh-CN",
    "tamil": "ta",
}

TARGET_FILES = []

MAX_ROWS_PER_FILE = 3000
SAMPLE_SEED = 42
BATCH_SIZE = 150
CHECKPOINT_EVERY_BATCHES = 5
MAX_WORKERS = 16


def translate_batch_fast(translator, text_batch):
    try:
        translated = translator.translate_batch(text_batch)
        if translated and len(translated) == len(text_batch):
            return translated
    except Exception:
        pass

    def _translate_one(txt):
        try:
            return translator.translate(txt)
        except Exception:
            return ""

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        return list(pool.map(_translate_one, text_batch))


all_files = [f"sentiment_{lang}.xlsx" for lang in lang_codes]
file_list = TARGET_FILES if TARGET_FILES else all_files

# Hard guard: never translate anxiety files in this sentiment-only notebook.
file_list = [f for f in file_list if "anxiety" not in f.lower()]

for filename in file_list:
    input_path = os.path.join(data_dir, filename)
    if not os.path.exists(input_path):
        print(f"Skipped missing file: {input_path}")
        continue

    base_name = os.path.splitext(filename)[0]
    language_name = base_name.split("_")[-1]
    target_lang = lang_codes.get(language_name)
    if target_lang is None:
        print(f"Skipped unknown language in filename: {filename}")
        continue

    output_path = os.path.join(data_dir, f"{base_name}_translated.xlsx")
    checkpoint_path = os.path.join(data_dir, f"{base_name}_translated_checkpoint.xlsx")

    df_src_full = pd.read_excel(input_path)
    if len(df_src_full) > MAX_ROWS_PER_FILE:
        df_src = df_src_full.sample(n=MAX_ROWS_PER_FILE, random_state=SAMPLE_SEED).reset_index(drop=True)
    elif len(df_src_full) < MAX_ROWS_PER_FILE:
        # Force exactly 3000 rows by upsampling with replacement when needed.
        df_src = df_src_full.sample(n=MAX_ROWS_PER_FILE, replace=True, random_state=SAMPLE_SEED).reset_index(drop=True)
    else:
        df_src = df_src_full.copy().reset_index(drop=True)

    if "text" not in df_src.columns:
        print(f"Skipped (no text column): {input_path}")
        continue

    df_src["text"] = df_src["text"].map(clean_text_value)
    df_src = df_src[df_src["text"].str.strip() != ""].reset_index(drop=True)

    # Re-expand to exactly 3000 after cleaning if empty texts were removed.
    if len(df_src) == 0:
        print(f"Skipped (all cleaned text empty): {input_path}")
        continue
    if len(df_src) < MAX_ROWS_PER_FILE:
        df_src = df_src.sample(n=MAX_ROWS_PER_FILE, replace=True, random_state=SAMPLE_SEED + 99).reset_index(drop=True)
    elif len(df_src) > MAX_ROWS_PER_FILE:
        df_src = df_src.sample(n=MAX_ROWS_PER_FILE, random_state=SAMPLE_SEED + 99).reset_index(drop=True)

    if os.path.exists(checkpoint_path):
        df_out = pd.read_excel(checkpoint_path)
        if "translated_text" not in df_out.columns or len(df_out) != len(df_src):
            df_out = df_src.copy()
            df_out["translated_text"] = ""
        else:
            df_out["text"] = df_src["text"]
    else:
        df_out = df_src.copy()
        df_out["translated_text"] = ""

    translator = GoogleTranslator(source="en", target=target_lang)

    pending_idx = df_out.index[df_out["translated_text"].astype(str).str.strip() == ""].tolist()
    print(f"Translating {filename}: {len(pending_idx)} rows remaining (fixed total {MAX_ROWS_PER_FILE})")

    batch_counter = 0
    for i in range(0, len(pending_idx), BATCH_SIZE):
        idx_batch = pending_idx[i:i + BATCH_SIZE]
        text_batch = df_out.loc[idx_batch, "text"].fillna("").astype(str).tolist()

        translated_batch = translate_batch_fast(translator, text_batch)

        for idx_val, tr in zip(idx_batch, translated_batch):
            df_out.at[idx_val, "translated_text"] = tr

        batch_counter += 1

        if batch_counter % CHECKPOINT_EVERY_BATCHES == 0:
            df_out.to_excel(checkpoint_path, index=False)
            print(f"Checkpoint saved: {checkpoint_path}")

    final_save = df_out.copy()
    translated_mask = final_save["translated_text"].astype(str).str.strip() != ""
    final_save = final_save.loc[translated_mask].copy()
    final_save["text"] = final_save["translated_text"]
    final_save = final_save.drop(columns=["translated_text"])
    final_save.to_excel(output_path, index=False)

    df_out.to_excel(checkpoint_path, index=False)

    print(f"Completed and saved (translated rows only): {output_path}")

print("Sentiment translation run finished.")

Translating sentiment_bengali.xlsx: 3000 rows remaining (fixed total 3000)
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali_translated_checkpoint.xlsx
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali_translated_checkpoint.xlsx
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali_translated_checkpoint.xlsx
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali_translated_checkpoint.xlsx
Completed and saved (translated rows only): C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_bengali_translated.xlsx
Translating sentiment_chinese.xlsx: 3000 rows remaining (fixed total 3000)
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_chinese_translated_checkpoint.xlsx
Checkpoint saved: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\data\sentiment_chinese_translated_chec

In [28]:
# Create sentiment-only training datasets for base model and LoRA experiments
# Outputs:
# 1) combined_basemodel_df
# 2) english_lora_df
# 3) bengali_lora_df
# 4) chinese_lora_df
# 5) tamil_lora_df

languages = ["english", "bengali", "chinese", "tamil"]
output_dir = os.path.join(os.path.dirname(df_sentiment_path), "data")


def _prepare_sentiment_df(df, language_name):
    work = df.copy()
    if "text" not in work.columns:
        raise ValueError(f"Missing 'text' column for sentiment-{language_name}")
    if "label" not in work.columns:
        raise ValueError(f"Missing 'label' column for sentiment-{language_name}")

    work = work[["text", "label"]].copy()
    work["language"] = language_name
    return work.reset_index(drop=True)


def _load_or_get_sentiment(language_name):
    if "sentiment_by_language" in globals() and isinstance(globals()["sentiment_by_language"], dict) and language_name in globals()["sentiment_by_language"]:
        return globals()["sentiment_by_language"][language_name].copy()

    file_path = os.path.join(output_dir, f"sentiment_{language_name}.xlsx")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing file: {file_path}. Run the split cell first.")
    return pd.read_excel(file_path)


per_language_sentiment = {}
for lang in languages:
    sentiment_df_lang = _prepare_sentiment_df(_load_or_get_sentiment(lang), lang)
    per_language_sentiment[lang] = sentiment_df_lang

combined_basemodel_df = pd.concat([per_language_sentiment[lang] for lang in languages], ignore_index=True)
english_lora_df = per_language_sentiment["english"].copy()
bengali_lora_df = per_language_sentiment["bengali"].copy()
chinese_lora_df = per_language_sentiment["chinese"].copy()
tamil_lora_df = per_language_sentiment["tamil"].copy()

print("combined_basemodel_df:", combined_basemodel_df.shape)
print("english_lora_df:", english_lora_df.shape)
print("bengali_lora_df:", bengali_lora_df.shape)
print("chinese_lora_df:", chinese_lora_df.shape)
print("tamil_lora_df:", tamil_lora_df.shape)

combined_basemodel_df.head()

combined_basemodel_df: (12000, 3)
english_lora_df: (3000, 3)
bengali_lora_df: (3000, 3)
chinese_lora_df: (3000, 3)
tamil_lora_df: (3000, 3)


,text,label,language
0,YPG has carried out numerous attacks. Yet you ...,sadness,english
1,Well that's why I'm reading up on the scientif...,neutral,english
2,"Yes, listening to loud noises can damage your ...",fear,english
3,"Whoa, should've said that in the post. Threats...",fear,english
4,I'm 25 and I still don't have a license. I'm t...,fear,english


In [29]:
# Save/export the 5 pre-split datasets to cleaned_data/
project_root = os.getcwd()
cleaned_data_dir = os.path.join(project_root, "cleaned_data")
os.makedirs(cleaned_data_dir, exist_ok=True)

pre_split_datasets = {
    "combined_basemodel_df": combined_basemodel_df,
    "english_lora_df": english_lora_df,
    "bengali_lora_df": bengali_lora_df,
    "chinese_lora_df": chinese_lora_df,
    "tamil_lora_df": tamil_lora_df,
}

export_summary = []
for name, df in pre_split_datasets.items():
    csv_name = f"{name}.csv"
    out_path = os.path.join(cleaned_data_dir, csv_name)

    df.to_csv(out_path, index=False, encoding="utf-8")
    export_summary.append((name, len(df), out_path))

for name, n_rows, out_path in export_summary:
    print(f"{name}: {n_rows} rows")
    print(f"  export: {out_path}")

combined_basemodel_df: 12000 rows
  export: c:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\combined_basemodel_df.csv
english_lora_df: 3000 rows
  export: c:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\english_lora_df.csv
bengali_lora_df: 3000 rows
  export: c:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\bengali_lora_df.csv
chinese_lora_df: 3000 rows
  export: c:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\chinese_lora_df.csv
tamil_lora_df: 3000 rows
  export: c:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\tamil_lora_df.csv


In [30]:
# Create train/validation/test splits for each sentiment dataset
import subprocess
import sys

try:
    from sklearn.model_selection import train_test_split
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TEST_SIZE = 0.15
VALID_SIZE = 0.15


def normalize_to_sentiment_schema(df):
    prepared = df.copy().reset_index(drop=True)

    # Backward compatibility for old multitask exports.
    if "task" in prepared.columns:
        prepared = prepared[prepared["task"].astype(str) == "sentiment"].copy()

    if "label" not in prepared.columns and "target" in prepared.columns:
        prepared["label"] = prepared["target"].astype(str)

    if "label" not in prepared.columns:
        raise ValueError("Expected a 'label' column for sentiment-only split.")

    if "text" not in prepared.columns:
        raise ValueError("Expected a 'text' column for sentiment-only split.")

    keep_cols = [c for c in ["text", "label", "language"] if c in prepared.columns]
    prepared = prepared[keep_cols].copy()
    prepared = prepared.dropna(subset=["text", "label"]).reset_index(drop=True)
    return prepared


def split_dataset(df, seed=42, test_size=0.15, valid_size=0.15):
    prepared = normalize_to_sentiment_schema(df)
    stratify_key = prepared["label"].astype(str)

    train_val_df, test_df = train_test_split(
        prepared,
        test_size=test_size,
        random_state=seed,
        stratify=stratify_key,
    )

    valid_share_of_train_val = valid_size / (1.0 - test_size)
    stratify_train_val = train_val_df["label"].astype(str)

    train_df, valid_df = train_test_split(
        train_val_df,
        test_size=valid_share_of_train_val,
        random_state=seed,
        stratify=stratify_train_val,
    )

    return {
        "train": train_df.reset_index(drop=True),
        "valid": valid_df.reset_index(drop=True),
        "test": test_df.reset_index(drop=True),
    }


project_root = os.path.dirname(df_sentiment_path)
cleaned_data_dir = os.path.join(project_root, "cleaned_data")


def load_dataset_with_fallback(dataset_key, fallback_var_name):
    csv_path = os.path.join(cleaned_data_dir, f"{fallback_var_name}.csv")
    if os.path.exists(csv_path):
        print(f"Loaded {dataset_key} from file: {csv_path}")
        return pd.read_csv(csv_path)

    if fallback_var_name in globals():
        print(f"File not found for {dataset_key}, using in-memory DataFrame: {fallback_var_name}")
        return globals()[fallback_var_name].copy()

    raise FileNotFoundError(
        f"Missing dataset for {dataset_key}. Expected file at {csv_path} or variable {fallback_var_name} in memory."
    )


dataset_map = {
    "combined_basemodel": load_dataset_with_fallback("combined_basemodel", "combined_basemodel_df"),
    "english_lora": load_dataset_with_fallback("english_lora", "english_lora_df"),
    "bengali_lora": load_dataset_with_fallback("bengali_lora", "bengali_lora_df"),
    "chinese_lora": load_dataset_with_fallback("chinese_lora", "chinese_lora_df"),
    "tamil_lora": load_dataset_with_fallback("tamil_lora", "tamil_lora_df"),
}

split_map = {
    name: split_dataset(df, seed=RANDOM_SEED, test_size=TEST_SIZE, valid_size=VALID_SIZE)
    for name, df in dataset_map.items()
}

combined_basemodel_train_df = split_map["combined_basemodel"]["train"]
combined_basemodel_valid_df = split_map["combined_basemodel"]["valid"]
combined_basemodel_test_df = split_map["combined_basemodel"]["test"]

english_lora_train_df = split_map["english_lora"]["train"]
english_lora_valid_df = split_map["english_lora"]["valid"]
english_lora_test_df = split_map["english_lora"]["test"]

bengali_lora_train_df = split_map["bengali_lora"]["train"]
bengali_lora_valid_df = split_map["bengali_lora"]["valid"]
bengali_lora_test_df = split_map["bengali_lora"]["test"]

chinese_lora_train_df = split_map["chinese_lora"]["train"]
chinese_lora_valid_df = split_map["chinese_lora"]["valid"]
chinese_lora_test_df = split_map["chinese_lora"]["test"]

tamil_lora_train_df = split_map["tamil_lora"]["train"]
tamil_lora_valid_df = split_map["tamil_lora"]["valid"]
tamil_lora_test_df = split_map["tamil_lora"]["test"]

for name, parts in split_map.items():
    print(f"{name}: train={parts['train'].shape}, valid={parts['valid'].shape}, test={parts['test'].shape}")

combined_basemodel_train_df.head()

Loaded combined_basemodel from file: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\combined_basemodel_df.csv
Loaded english_lora from file: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\english_lora_df.csv
Loaded bengali_lora from file: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\bengali_lora_df.csv
Loaded chinese_lora from file: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\chinese_lora_df.csv
Loaded tamil_lora from file: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\tamil_lora_df.csv
combined_basemodel: train=(8399, 3), valid=(1801, 3), test=(1800, 3)
english_lora: train=(2099, 3), valid=(451, 3), test=(450, 3)
bengali_lora: train=(2099, 3), valid=(451, 3), test=(450, 3)
chinese_lora: train=(2099, 3), valid=(451, 3), test=(450, 3)
tamil_lora: train=(2099, 3), valid=(451, 3), test=(450, 3)


,text,label,language
0,Yeah but being happy at other misfortune is so...,sadness,bengali
1,Let her contact you if she wants. Don’t bug he...,neutral,bengali
2,I have seen the movie Nosferatu. Believe it or...,neutral,tamil
3,Don't even bother with him lol. He's a mindles...,nervousness,tamil
4,I'm generally a supporter of law enforcement.....,neutral,tamil


In [31]:
# Sentiment-only training utilities (base model + LoRA adapters)
import os
import json
import subprocess
import sys

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

try:
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

try:
    from peft import LoraConfig, get_peft_model
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "peft"])
    from peft import LoraConfig, get_peft_model

try:
    from tqdm.auto import tqdm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "tqdm"])
    from tqdm.auto import tqdm


BASE_MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 192
BATCH_SIZE = 16
BASE_EPOCHS = 2
LORA_EPOCHS = 2
LEARNING_RATE_BASE = 2e-5
LEARNING_RATE_LORA = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_root = os.path.join(os.path.dirname(df_sentiment_path), "model")
base_model_dir = os.path.join(model_root, "base_model")
lora_root_dir = os.path.join(model_root, "lora_adapters")
os.makedirs(base_model_dir, exist_ok=True)
os.makedirs(lora_root_dir, exist_ok=True)

# Sentiment label mapping is learned from your combined dataset.
sentiment_labels = sorted(pd.Series(combined_basemodel_df["label"].astype(str)).unique().tolist())
sentiment_label2id = {label: i for i, label in enumerate(sentiment_labels)}
sentiment_id2label = {i: label for label, i in sentiment_label2id.items()}
print("Sentiment labels:", sentiment_label2id)


class SentimentTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=192):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text"])
        label_cls = sentiment_label2id[str(row["label"])]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels_cls": torch.tensor(label_cls, dtype=torch.long),
        }


class SentimentModel(nn.Module):
    def __init__(self, encoder, hidden_size, num_sentiment_labels):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(0.1)
        self.sentiment_head = nn.Linear(hidden_size, num_sentiment_labels)
        self.ce_loss = nn.CrossEntropyLoss()

    def forward(self, input_ids, attention_mask, labels_cls=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)
        logits_cls = self.sentiment_head(pooled)

        loss = None
        if labels_cls is not None:
            loss = self.ce_loss(logits_cls, labels_cls)

        return {
            "loss": loss,
            "logits_cls": logits_cls,
        }


def run_epoch(model, dataloader, optimizer=None, scheduler=None, desc=None, show_progress=True):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_steps = 0

    iterator = dataloader
    if show_progress:
        default_desc = "train" if train_mode else "eval"
        iterator = tqdm(dataloader, desc=desc or default_desc, leave=False)

    for batch in iterator:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_cls = batch["labels_cls"].to(device)

        with torch.set_grad_enabled(train_mode):
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels_cls=labels_cls,
            )
            loss = out["loss"]

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                if scheduler is not None:
                    scheduler.step()

        total_loss += float(loss.item())
        total_steps += 1

        if show_progress:
            iterator.set_postfix(loss=f"{(total_loss / max(total_steps, 1)):.4f}")

    return total_loss / max(total_steps, 1)


def build_dataloader(df, tokenizer, batch_size=16, shuffle=False):
    ds = SentimentTextDataset(df, tokenizer=tokenizer, max_length=MAX_LENGTH)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

Using device: cpu
Sentiment labels: {'fear': 0, 'grief': 1, 'nervousness': 2, 'neutral': 3, 'sadness': 4}


# Base Model training + LoRA adapter training with trained based model

In [32]:
# Train base model first, then train LoRA adapters from the trained base encoder (sentiment-only)

try:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def build_optimizer_and_scheduler(model, train_loader, lr, epochs):
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = max(1, len(train_loader) * epochs)
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    return optimizer, scheduler


def save_pretrained_with_retry(obj, save_dir, is_model=False, retries=3):
    os.makedirs(save_dir, exist_ok=True)
    last_err = None

    for attempt in range(1, retries + 1):
        try:
            if is_model:
                obj.save_pretrained(save_dir, safe_serialization=False)
            else:
                obj.save_pretrained(save_dir)
            return
        except Exception as e:
            last_err = e
            if attempt < retries:
                import gc
                import time

                gc.collect()
                time.sleep(float(attempt))

    raise last_err


def evaluate_sentiment_model(model, dataloader, desc="evaluation", show_progress=True):
    model.eval()

    cls_true, cls_pred = [], []
    loss_values = []

    iterator = dataloader
    if show_progress:
        iterator = tqdm(dataloader, desc=desc, leave=False)

    for batch in iterator:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_cls = batch["labels_cls"].to(device)

        with torch.no_grad():
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels_cls=labels_cls,
            )

        if out["loss"] is not None:
            loss_values.append(float(out["loss"].item()))

        logits_cls = out["logits_cls"].detach().cpu()
        labels_cls_cpu = labels_cls.detach().cpu()

        cls_true.extend(labels_cls_cpu.tolist())
        cls_pred.extend(logits_cls.argmax(dim=-1).tolist())

    metrics = {
        "loss": float(np.mean(loss_values)) if loss_values else np.nan,
        "accuracy": np.nan,
        "f1": np.nan,
        "precision": np.nan,
        "recall": np.nan,
    }

    if len(cls_true) > 0:
        metrics["accuracy"] = float(accuracy_score(cls_true, cls_pred))
        metrics["f1"] = float(f1_score(cls_true, cls_pred, average="macro", zero_division=0))
        metrics["precision"] = float(precision_score(cls_true, cls_pred, average="macro", zero_division=0))
        metrics["recall"] = float(recall_score(cls_true, cls_pred, average="macro", zero_division=0))

    return metrics


required_dfs = [
    "combined_basemodel_train_df", "combined_basemodel_valid_df", "combined_basemodel_test_df",
    "english_lora_train_df", "english_lora_valid_df",
    "bengali_lora_train_df", "bengali_lora_valid_df",
    "chinese_lora_train_df", "chinese_lora_valid_df",
    "tamil_lora_train_df", "tamil_lora_valid_df",
]
missing_dfs = [name for name in required_dfs if name not in globals()]
if missing_dfs:
    raise ValueError(f"Missing split DataFrames. Run the split cell first. Missing: {missing_dfs}")


# ---------- 1) Base model training ----------
trained_encoder_dir = os.path.join(base_model_dir, "encoder")
task_heads_path = os.path.join(base_model_dir, "task_heads.pt")
best_base_checkpoint = os.path.join(base_model_dir, "best_sentiment_model.pt")
legacy_best_checkpoint = os.path.join(base_model_dir, "best_multitask_model.pt")
base_epoch_ckpt_dir = os.path.join(base_model_dir, "checkpoints")
os.makedirs(base_epoch_ckpt_dir, exist_ok=True)

base_artifacts_ready = (
    os.path.exists(best_base_checkpoint)
    and os.path.isdir(trained_encoder_dir)
    and os.path.exists(task_heads_path)
)

if base_artifacts_ready:
    print(f"[BASE] Sentiment base model already exists. Loading from: {trained_encoder_dir}")
    tokenizer = AutoTokenizer.from_pretrained(trained_encoder_dir)
    base_encoder = AutoModel.from_pretrained(trained_encoder_dir)
else:
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
    base_encoder = AutoModel.from_pretrained(BASE_MODEL_NAME)

train_loader_base = build_dataloader(combined_basemodel_train_df, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
valid_loader_base = build_dataloader(combined_basemodel_valid_df, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

hidden_size = base_encoder.config.hidden_size
base_model = SentimentModel(base_encoder, hidden_size, len(sentiment_label2id)).to(device)

optimizer_base, scheduler_base = build_optimizer_and_scheduler(
    base_model,
    train_loader_base,
    lr=LEARNING_RATE_BASE,
    epochs=BASE_EPOCHS,
)

best_val_f1 = -1.0
start_epoch = 1
resume_checkpoint = None

epoch_ckpt_candidates = []
for name in os.listdir(base_epoch_ckpt_dir):
    if name.startswith("sentiment_base_epoch_") and name.endswith(".pt"):
        epoch_str = name[len("sentiment_base_epoch_"):-len(".pt")]
        if epoch_str.isdigit():
            epoch_ckpt_candidates.append((int(epoch_str), os.path.join(base_epoch_ckpt_dir, name)))

if epoch_ckpt_candidates:
    epoch_ckpt_candidates.sort(key=lambda x: x[0])
    latest_epoch, resume_checkpoint = epoch_ckpt_candidates[-1]
    start_epoch = latest_epoch + 1

if base_artifacts_ready:
    base_model.load_state_dict(torch.load(best_base_checkpoint, map_location=device))
    start_epoch = BASE_EPOCHS + 1
elif resume_checkpoint is not None and start_epoch <= BASE_EPOCHS:
    print(f"[BASE] Resuming from: {resume_checkpoint} (starting at epoch {start_epoch})")
    base_model.load_state_dict(torch.load(resume_checkpoint, map_location=device))
    resume_metrics = evaluate_sentiment_model(
        base_model,
        valid_loader_base,
        desc="[BASE] resume-valid baseline",
        show_progress=True,
    )
    best_val_f1 = float(resume_metrics.get("f1", np.nan))
    if np.isnan(best_val_f1):
        best_val_f1 = -1.0
elif os.path.exists(legacy_best_checkpoint) and not os.path.exists(best_base_checkpoint):
    print(f"[BASE] Migrating legacy checkpoint: {legacy_best_checkpoint}")
    base_model.load_state_dict(torch.load(legacy_best_checkpoint, map_location=device))
    torch.save(base_model.state_dict(), best_base_checkpoint)
    start_epoch = BASE_EPOCHS + 1

if start_epoch <= BASE_EPOCHS:
    for epoch in range(start_epoch, BASE_EPOCHS + 1):
        train_loss = run_epoch(
            base_model,
            train_loader_base,
            optimizer_base,
            scheduler_base,
            desc=f"[BASE] train {epoch}/{BASE_EPOCHS}",
            show_progress=True,
        )
        valid_loss = run_epoch(
            base_model,
            valid_loader_base,
            optimizer=None,
            scheduler=None,
            desc=f"[BASE] valid {epoch}/{BASE_EPOCHS}",
            show_progress=True,
        )
        valid_metrics = evaluate_sentiment_model(
            base_model,
            valid_loader_base,
            desc=f"[BASE] valid-metrics {epoch}/{BASE_EPOCHS}",
            show_progress=True,
        )
        valid_f1 = float(valid_metrics.get("f1", np.nan))
        if np.isnan(valid_f1):
            valid_f1 = -1.0
        print(f"[BASE] epoch={epoch}/{BASE_EPOCHS} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} valid_f1={valid_f1:.4f}")

        epoch_ckpt_path = os.path.join(base_epoch_ckpt_dir, f"sentiment_base_epoch_{epoch}.pt")
        torch.save(base_model.state_dict(), epoch_ckpt_path)

        if valid_f1 > best_val_f1:
            best_val_f1 = valid_f1
            torch.save(base_model.state_dict(), best_base_checkpoint)

    if not os.path.exists(best_base_checkpoint):
        torch.save(base_model.state_dict(), best_base_checkpoint)

    base_model.load_state_dict(torch.load(best_base_checkpoint, map_location=device))
else:
    print("[BASE] Base training already complete for configured BASE_EPOCHS.")

os.makedirs(trained_encoder_dir, exist_ok=True)
save_pretrained_with_retry(base_model.encoder, trained_encoder_dir, is_model=True)
save_pretrained_with_retry(tokenizer, trained_encoder_dir, is_model=False)

torch.save(
    {
        "sentiment_head": base_model.sentiment_head.state_dict(),
    },
    task_heads_path,
)

with open(os.path.join(base_model_dir, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "sentiment_label2id": sentiment_label2id,
            "sentiment_id2label": sentiment_id2label,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved sentiment base model artifacts to: {base_model_dir}")


# ---------- 2) LoRA adapter training (from trained base encoder) ----------
lora_train_map = {
    "english": (english_lora_train_df, english_lora_valid_df),
    "bengali": (bengali_lora_train_df, bengali_lora_valid_df),
    "chinese": (chinese_lora_train_df, chinese_lora_valid_df),
    "tamil": (tamil_lora_train_df, tamil_lora_valid_df),
}

lora_training_log = {}
heads_state = torch.load(task_heads_path, map_location="cpu")

for lang, (train_df, valid_df) in lora_train_map.items():
    adapter_dir = os.path.join(lora_root_dir, lang)
    os.makedirs(adapter_dir, exist_ok=True)

    adapter_cfg_path = os.path.join(adapter_dir, "adapter_config.json")
    adapter_heads_path = os.path.join(adapter_dir, "task_heads.pt")

    if os.path.exists(adapter_cfg_path) and os.path.exists(adapter_heads_path):
        print(f"[LoRA-{lang}] Training already complete. Skipping: {adapter_dir}")
        lora_training_log[lang] = {"best_valid_f1": None, "adapter_dir": adapter_dir, "skipped": True}
        continue

    print(f"\n[LoRA-{lang}] preparing training")

    train_loader = build_dataloader(train_df, tokenizer, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = build_dataloader(valid_df, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

    encoder_for_lora = AutoModel.from_pretrained(trained_encoder_dir)

    lora_cfg = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        target_modules=["query", "key", "value"],
        bias="none",
        task_type="FEATURE_EXTRACTION",
    )

    peft_encoder = get_peft_model(encoder_for_lora, lora_cfg)
    lora_model = SentimentModel(peft_encoder, peft_encoder.config.hidden_size, len(sentiment_label2id)).to(device)

    lora_model.sentiment_head.load_state_dict(heads_state["sentiment_head"])

    optimizer_lora, scheduler_lora = build_optimizer_and_scheduler(
        lora_model,
        train_loader,
        lr=LEARNING_RATE_LORA,
        epochs=LORA_EPOCHS,
    )

    best_lora_f1 = -1.0
    best_adapter_state = None
    best_head_state = None

    for epoch in range(1, LORA_EPOCHS + 1):
        train_loss = run_epoch(
            lora_model,
            train_loader,
            optimizer_lora,
            scheduler_lora,
            desc=f"[LoRA-{lang}] train {epoch}/{LORA_EPOCHS}",
            show_progress=True,
        )
        valid_loss = run_epoch(
            lora_model,
            valid_loader,
            optimizer=None,
            scheduler=None,
            desc=f"[LoRA-{lang}] valid {epoch}/{LORA_EPOCHS}",
            show_progress=True,
        )
        valid_metrics = evaluate_sentiment_model(
            lora_model,
            valid_loader,
            desc=f"[LoRA-{lang}] valid-metrics {epoch}/{LORA_EPOCHS}",
            show_progress=True,
        )
        valid_f1 = float(valid_metrics.get("f1", np.nan))
        if np.isnan(valid_f1):
            valid_f1 = -1.0
        print(f"[LoRA-{lang}] epoch={epoch}/{LORA_EPOCHS} train_loss={train_loss:.4f} valid_loss={valid_loss:.4f} valid_f1={valid_f1:.4f}")

        if valid_f1 > best_lora_f1:
            best_lora_f1 = valid_f1
            best_adapter_state = {k: v.detach().cpu().clone() for k, v in lora_model.encoder.state_dict().items()}
            best_head_state = {k: v.detach().cpu().clone() for k, v in lora_model.sentiment_head.state_dict().items()}

    if best_adapter_state is None:
        best_adapter_state = {k: v.detach().cpu().clone() for k, v in lora_model.encoder.state_dict().items()}
        best_head_state = {k: v.detach().cpu().clone() for k, v in lora_model.sentiment_head.state_dict().items()}

    lora_model.encoder.load_state_dict(best_adapter_state)
    lora_model.sentiment_head.load_state_dict(best_head_state)

    legacy_full_ckpt = os.path.join(adapter_dir, "best_lora_full_model.pt")
    if os.path.exists(legacy_full_ckpt):
        os.remove(legacy_full_ckpt)

    save_pretrained_with_retry(lora_model.encoder, adapter_dir, is_model=True)
    torch.save(
        {
            "sentiment_head": lora_model.sentiment_head.state_dict(),
        },
        adapter_heads_path,
    )

    with open(os.path.join(adapter_dir, "adapter_meta.json"), "w", encoding="utf-8") as f:
        json.dump(
            {
                "language": lang,
                "base_encoder_path": trained_encoder_dir,
                "best_valid_f1": float(best_lora_f1) if best_lora_f1 >= 0 else None,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )

    lora_training_log[lang] = {
        "best_valid_f1": float(best_lora_f1) if best_lora_f1 >= 0 else None,
        "adapter_dir": adapter_dir,
        "skipped": False,
    }
    print(f"[LoRA-{lang}] saved adapter to: {adapter_dir}")

print("\nSentiment-only training complete.")
print("Base model dir:", base_model_dir)
print("LoRA adapters root:", lora_root_dir)
for lang, info in lora_training_log.items():
    print(f"  {lang}: {info}")

[BASE] epoch=1/2 train_loss=0.8615 valid_loss=0.5937 valid_f1=0.4521


[BASE] epoch=2/2 train_loss=0.5513 valid_loss=0.5543 valid_f1=0.4759
Saved sentiment base model artifacts to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\base_model

[LoRA-english] preparing training


[LoRA-english] epoch=1/2 train_loss=0.5499 valid_loss=0.6098 valid_f1=0.4736


[LoRA-english] epoch=2/2 train_loss=0.5454 valid_loss=0.6099 valid_f1=0.4833
[LoRA-english] saved adapter to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\lora_adapters\english

[LoRA-bengali] preparing training


[LoRA-bengali] epoch=1/2 train_loss=0.4973 valid_loss=0.5050 valid_f1=0.5958


[LoRA-bengali] epoch=2/2 train_loss=0.4818 valid_loss=0.5087 valid_f1=0.5920
[LoRA-bengali] saved adapter to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\lora_adapters\bengali

[LoRA-chinese] preparing training


[LoRA-chinese] epoch=1/2 train_loss=0.5044 valid_loss=0.4924 valid_f1=0.5989


[LoRA-chinese] epoch=2/2 train_loss=0.4752 valid_loss=0.4868 valid_f1=0.6151
[LoRA-chinese] saved adapter to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\lora_adapters\chinese

[LoRA-tamil] preparing training


[LoRA-tamil] epoch=1/2 train_loss=0.4576 valid_loss=0.4216 valid_f1=0.4914


[LoRA-tamil] epoch=2/2 train_loss=0.4581 valid_loss=0.4105 valid_f1=0.5010
[LoRA-tamil] saved adapter to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\lora_adapters\tamil

Sentiment-only training complete.
Base model dir: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\base_model
LoRA adapters root: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\model\lora_adapters
  english: {'best_valid_f1': 0.48327764998319334, 'adapter_dir': 'C:\\Users\\caleb\\OneDrive\\Desktop\\Y4S2\\DSA4262\\project\\model\\lora_adapters\\english', 'skipped': False}
  bengali: {'best_valid_f1': 0.5957789735017458, 'adapter_dir': 'C:\\Users\\caleb\\OneDrive\\Desktop\\Y4S2\\DSA4262\\project\\model\\lora_adapters\\bengali', 'skipped': False}
  chinese: {'best_valid_f1': 0.6151241433845833, 'adapter_dir': 'C:\\Users\\caleb\\OneDrive\\Desktop\\Y4S2\\DSA4262\\project\\model\\lora_adapters\\chinese', 'skipped': False}
  tamil: {'best_valid_f1': 0.5009917036479767, 'adapter_dir': 'C:\\Users

In [ ]:
# Evaluate base model vs LoRA adapters on the 4 language test sets (single combined cell)
try:
    from peft import PeftModel
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "peft"])
    from peft import PeftModel

try:
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


def evaluate_sentiment_model(model, dataloader, desc="evaluation", show_progress=True):
    model.eval()

    cls_true, cls_pred = [], []
    loss_values = []

    iterator = dataloader
    if show_progress:
        iterator = tqdm(dataloader, desc=desc, leave=False)

    for batch in iterator:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_cls = batch["labels_cls"].to(device)

        with torch.no_grad():
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels_cls=labels_cls,
            )

        if out["loss"] is not None:
            loss_values.append(float(out["loss"].item()))

        logits_cls = out["logits_cls"].detach().cpu()
        labels_cls_cpu = labels_cls.detach().cpu()

        cls_true.extend(labels_cls_cpu.tolist())
        cls_pred.extend(logits_cls.argmax(dim=-1).tolist())

    metrics = {
        "loss": float(np.mean(loss_values)) if loss_values else np.nan,
        "accuracy": np.nan,
        "f1": np.nan,
        "precision": np.nan,
        "recall": np.nan,
    }

    if len(cls_true) > 0:
        metrics["accuracy"] = float(accuracy_score(cls_true, cls_pred))
        metrics["f1"] = float(f1_score(cls_true, cls_pred, average="macro", zero_division=0))
        metrics["precision"] = float(precision_score(cls_true, cls_pred, average="macro", zero_division=0))
        metrics["recall"] = float(recall_score(cls_true, cls_pred, average="macro", zero_division=0))

    return metrics


def load_base_model_for_eval():
    encoder_dir = os.path.join(base_model_dir, "encoder")
    checkpoint_path = os.path.join(base_model_dir, "best_sentiment_model.pt")
    if not os.path.exists(checkpoint_path):
        checkpoint_path = os.path.join(base_model_dir, "best_multitask_model.pt")

    base_encoder_eval = AutoModel.from_pretrained(encoder_dir)
    model_eval = SentimentModel(base_encoder_eval, base_encoder_eval.config.hidden_size, len(sentiment_label2id)).to(device)
    model_eval.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model_eval.eval()
    return model_eval


def load_lora_model_for_eval(lang):
    encoder_dir = os.path.join(base_model_dir, "encoder")
    adapter_dir = os.path.join(lora_root_dir, lang)

    base_encoder_eval = AutoModel.from_pretrained(encoder_dir)
    peft_encoder_eval = PeftModel.from_pretrained(base_encoder_eval, adapter_dir)

    model_eval = SentimentModel(peft_encoder_eval, peft_encoder_eval.config.hidden_size, len(sentiment_label2id)).to(device)

    heads_path = os.path.join(adapter_dir, "task_heads.pt")
    heads_state = torch.load(heads_path, map_location=device)
    model_eval.sentiment_head.load_state_dict(heads_state["sentiment_head"])
    model_eval.eval()
    return model_eval


language_test_map = {
    "english": english_lora_test_df,
    "bengali": bengali_lora_test_df,
    "chinese": chinese_lora_test_df,
    "tamil": tamil_lora_test_df,
}

base_eval_model = load_base_model_for_eval()
rows = []

for lang, test_df in language_test_map.items():
    test_loader = build_dataloader(test_df, tokenizer, batch_size=BATCH_SIZE, shuffle=False)

    metrics_base = evaluate_sentiment_model(
        base_eval_model,
        test_loader,
        desc=f"[EVAL] base on {lang}_lora_test",
        show_progress=True,
    )
    rows.append({
        "model": "base",
        "dataset": f"{lang}_lora_test",
        **metrics_base,
    })

    lora_eval_model = load_lora_model_for_eval(lang)
    metrics_lora = evaluate_sentiment_model(
        lora_eval_model,
        test_loader,
        desc=f"[EVAL] lora_{lang} on {lang}_lora_test",
        show_progress=True,
    )
    rows.append({
        "model": f"lora_{lang}",
        "dataset": f"{lang}_lora_test",
        **metrics_lora,
    })

comparison_sentiment_df = pd.DataFrame(rows)
comparison_sentiment_df = comparison_sentiment_df[["model", "dataset", "loss", "accuracy", "f1", "precision", "recall"]]


# Keep compatibility for export cell.
comparison_df = comparison_sentiment_df.copy()

print("Combined evaluation complete (language-only, anxiety treated as binary).")
display(comparison_sentiment_df.sort_values(["dataset", "model"]).reset_index(drop=True))

Combined evaluation complete (language-only, anxiety treated as binary).


,model,dataset,accuracy,f1,precision,recall
0,base,english_lora_test,0.878889,0.671532,0.697925,0.663029
1,lora_english,english_lora_test,0.883333,0.681564,0.694552,0.680514
2,base,bengali_lora_test,0.920000,0.573066,0.559127,0.590776
3,lora_bengali,bengali_lora_test,0.926667,0.574648,0.565387,0.584737
4,base,chinese_lora_test,0.926667,0.634027,0.704499,0.631646
5,lora_chinese,chinese_lora_test,0.923333,0.566438,0.563705,0.573266
6,base,tamil_lora_test,0.920000,0.581687,0.574433,0.591514
7,lora_tamil,tamil_lora_test,0.923333,0.591712,0.579628,0.609371


In [ ]:
# Export sentiment-only evaluation metrics table from previous cell
metrics_out_dir = os.path.join(os.path.dirname(df_sentiment_path), "cleaned_data")
os.makedirs(metrics_out_dir, exist_ok=True)

if "comparison_sentiment_df" in globals():
    export_df = comparison_sentiment_df
else:
    raise ValueError("No sentiment comparison dataframe found. Run the sentiment evaluation cell first.")

metrics_out_path = os.path.join(metrics_out_dir, "model_comparison_metrics_sentiment.csv")
export_df.to_csv(metrics_out_path, index=False, encoding="utf-8")

print(f"Saved sentiment metrics to: {metrics_out_path}")
export_df

Saved metrics to: C:\Users\caleb\OneDrive\Desktop\Y4S2\DSA4262\project\cleaned_data\model_comparison_metrics_combined_binary.csv


,model,dataset,accuracy,f1,precision,recall
0,base,english_lora_test,0.878889,0.671532,0.697925,0.663029
1,lora_english,english_lora_test,0.883333,0.681564,0.694552,0.680514
2,base,bengali_lora_test,0.920000,0.573066,0.559127,0.590776
3,lora_bengali,bengali_lora_test,0.926667,0.574648,0.565387,0.584737
4,base,chinese_lora_test,0.926667,0.634027,0.704499,0.631646
5,lora_chinese,chinese_lora_test,0.923333,0.566438,0.563705,0.573266
6,base,tamil_lora_test,0.920000,0.581687,0.574433,0.591514
7,lora_tamil,tamil_lora_test,0.923333,0.591712,0.579628,0.609371
